# **코스피 소형주 필터링(소형주 선별 작업)**

In [ ]:
import pandas as pd
import requests
from io import StringIO

dfs = []

for page in range(1, 50):
    url = f"https://finance.naver.com/sise/sise_market_sum.naver?sosok=0&page={page}"

    res = requests.get(url)
    res.encoding = 'euc-kr'

    tables = pd.read_html(StringIO(res.text))

    for table in tables:
        if '종목명' in table.columns:
            dfs.append(table)

df = pd.concat(dfs, ignore_index=True)

# 쓰레기 행 제거
df = df[df['종목명'].notna()]
df = df[df['종목명'] != '종목명']

# 필요한 컬럼
df = df[['종목명', '시가총액', 'PER']]

# 시가총액 변환
df['시가총액'] = df['시가총액'].astype(str).str.replace(',', '').astype(float)

# ETF 제거
etf_keywords = "KODEX|TIGER|KBSTAR|ARIRANG|KOSEF|HANARO|RISE|KIWOOM|ACE|SOL|PLUS|ETN|선물|인버스|레버리지"
df = df[~df['종목명'].str.contains(etf_keywords)]

# PER 없는 기업 제거
df = df[df['PER'].notna()]

df = df[df['PER'] > 0]

# 중복 제거
df = df.drop_duplicates(subset='종목명')

# 정렬
df = df.sort_values('시가총액', ascending=False)

# 소형주 (하위 30%)
small_cap = df.iloc[int(len(df)*0.7):]

pd.set_option('display.max_rows', None)
print(small_cap)
print("소형주 개수:", len(small_cap))

              종목명    시가총액      PER
1567      화승코퍼레이션  1401.0     3.92
1568       경동인베스트  1400.0     5.97
1573         HS애드  1395.0     8.19
1574        대성홀딩스  1395.0     5.55
1576          SK우  1390.0    11.23
1581          디씨엠  1385.0    11.74
1588       한국무브넥스  1360.0     4.56
1590      제일파마홀딩스  1355.0     9.26
1599    이지스레지던스리츠  1347.0     9.30
1604         아남전자  1344.0    22.93
1605           동방  1341.0     8.76
1613          디와이  1309.0     4.78
1615         HS화성  1305.0     3.93
1621        유니퀘스트  1305.0     6.32
1616           두올  1305.0     4.30
1623       경동도시가스  1300.0     3.96
1625        한국화장품  1290.0    28.08
1629           신흥  1290.0    28.52
1630       HL D&I  1285.0    10.48
1637         샘표식품  1272.0     6.38
1641      세이브존I&C  1260.0    12.53
1645         극동유화  1257.0    50.07
1647         조선선재  1248.0     9.70
1656           벽산  1238.0     6.93
1664        태경비케이  1233.0     4.62
1665         화천기계  1230.0    27.40
1671         DKME  1220.0    16.61
1689       수산세보틱스  1

# **코스피 소형주 스크랩**

In [ ]:
stock_data = [
("013520","화승코퍼레이션"),
("012320","경동인베스트"),
("035000","HS애드"),
("016710","대성홀딩스"),
("024090","디씨엠"),
("010100","한국무브넥스"),
("002620","제일파마홀딩스"),
("350520","이지스레지던스리츠"),
("008700","아남전자"),
("004140","동방"),
("013570","디와이"),
("002460","HS화성"),
("077500","유니퀘스트"),
("016740","두올"),
("267290","경동도시가스"),
("123690","한국화장품"),
("004080","신흥"),
("014790","HL D&I"),
("248170","샘표식품"),
("067830","세이브존I&C"),
("014530","극동유화"),
("120030","조선선재"),
("007210","벽산"),
("014580","태경비케이"),
("010660","화천기계"),
("015590","DKME"),
("017550","수산세보틱스"),
("012800","대창"),
("000230","일동홀딩스"),
("034590","인천도시가스"),
("020120","키다리스튜디오"),
("017900","광전자"),
("001270","부국증권우"),
("001560","제일연마"),
("004100","태양금속"),
("227840","현대코퍼레이션홀딩스"),
("267850","아시아나IDT"),
("002450","삼익악기"),
("058850","KTcs"),
("004840","DRB동일"),
("023450","동남합성"),
("006890","태경케미컬"),
("002720","국제약품"),
("023800","인지컨트롤스"),
("264900","크라운제과"),
("017370","우신시스템"),
("004060","SG세계물산"),
("005740","크라운해태홀딩스"),
("008260","NI스틸"),
("119650","KC코트렐"),
("005750","대림바스")
]

# **PER / PBR 먼저 선별**

In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from io import StringIO
import time

results = []

for code, name in stock_data:
    try:
        url = f"https://finance.naver.com/item/main.naver?code={code}"
        res = requests.get(url)
        res.encoding = 'euc-kr'

        soup = BeautifulSoup(res.text, 'html.parser')

        # PER / PBR
        per = soup.select_one("#_per")
        pbr = soup.select_one("#_pbr")

        per = per.text if per else None
        pbr = pbr.text if pbr else None


        tables = pd.read_html(StringIO(res.text))

        psr, pcr = None, None


        for table in tables:
            for i in range(len(table)):
                row = str(table.iloc[i, 0])

                if "PSR" in row:
                    psr = table.iloc[i, 1]
                if "PCR" in row:
                    pcr = table.iloc[i, 1]

        results.append([name, per, pbr, pcr, psr])

        time.sleep(0.3)

    except:
        continue

df_all = pd.DataFrame(results, columns=['종목명','PER','PBR','PCR','PSR'])

# 숫자 변환
df_all['PER'] = pd.to_numeric(df_all['PER'], errors='coerce')
df_all['PBR'] = pd.to_numeric(df_all['PBR'], errors='coerce')
df_all['PCR'] = pd.to_numeric(df_all['PCR'], errors='coerce')
df_all['PSR'] = pd.to_numeric(df_all['PSR'], errors='coerce')


df_all = df_all.dropna(subset=['PER','PBR'])

# 필터
filtered = df_all[
    (df_all['PER'] > 0) & (df_all['PER'] < 15) &
    (df_all['PBR'] > 0) & (df_all['PBR'] < 1)
]

top50 = filtered.nsmallest(50, 'PER')

print(top50)

           종목명    PER   PBR  PCR  PSR
28       일동홀딩스   1.73  0.87  NaN  NaN
35  현대코퍼레이션홀딩스   3.26  0.35  NaN  NaN
14      경동도시가스   3.98  0.27  NaN  NaN
0      화승코퍼레이션   3.99  0.52  NaN  NaN
45       우신시스템   4.07  0.60  NaN  NaN
13          두올   4.43  0.51  NaN  NaN
23       태경비케이   4.68  0.60  NaN  NaN
5       한국무브넥스   4.72  0.35  NaN  NaN
38        KTcs   4.72  0.46  NaN  NaN
11        HS화성   4.79  0.30  NaN  NaN
37        삼익악기   5.36  0.34  NaN  NaN
48        NI스틸   5.43  0.38  NaN  NaN
47    크라운해태홀딩스   5.51  0.30  NaN  NaN
3        대성홀딩스   5.59  0.31  NaN  NaN
43      인지컨트롤스   5.98  0.44  NaN  NaN
1       경동인베스트   6.11  0.28  NaN  NaN
29      인천도시가스   6.13  0.46  NaN  NaN
18        샘표식품   6.46  0.49  NaN  NaN
12       유니퀘스트   6.57  0.55  NaN  NaN
22          벽산   7.04  0.35  NaN  NaN
33        제일연마   8.17  0.72  NaN  NaN
2         HS애드   8.24  0.66  NaN  NaN
9           동방   8.82  0.78  NaN  NaN
36     아시아나IDT   8.90  0.65  NaN  NaN
39       DRB동일   9.22  0.26  NaN  NaN
6      제일파마홀

# **PSR / PCR 직접 계산 및 선별(순위까지)**

In [1]:
import yfinance as yf
import pandas as pd
import time


codes = [
"013520","012320","035000","016710","024090","010100","002620","350520",
"008700","004140","013570","002460","077500","016740","267290","123690",
"004080","014790","248170","067830","014530","120030","007210","014580",
"010660","015590","017550","012800","000230","034590","020120","017900",
"001270","001560","004100","227840","267850","002450","058850","004840",
"023450","006890","002720","023800","264900","017370","004060","005740",
"008260","119650","005750"
]



results = []

for code in codes:
    try:
        ticker = yf.Ticker(code + ".KS")

        info = ticker.info
        market_cap = info.get("marketCap", None)

        financials = ticker.financials
        cashflow = ticker.cashflow

        # 매출
        revenue = None
        if "Total Revenue" in financials.index:
            revenue = financials.loc["Total Revenue"].iloc[0]

        # 영업현금흐름
        operating_cf = None
        if "Operating Cash Flow" in cashflow.index:
            operating_cf = cashflow.loc["Operating Cash Flow"].iloc[0]

        # PSR / PCR 계산
        psr = None
        pcr = None

        if market_cap and revenue and revenue != 0:
            psr = market_cap / revenue

        if market_cap and operating_cf and operating_cf != 0:
            pcr = market_cap / operating_cf

        results.append([
            code,
            market_cap,
            revenue,
            operating_cf,
            psr,
            pcr
        ])

        time.sleep(0.5)

    except:
        continue


df = pd.DataFrame(results, columns=[
    "종목코드","시가총액","매출액","영업현금흐름","PSR","PCR"
])



df["PSR"] = pd.to_numeric(df["PSR"], errors="coerce")
df["PCR"] = pd.to_numeric(df["PCR"], errors="coerce")


df = df[
    (df["PSR"] > 0) & (df["PSR"] < 3) &
    (df["PCR"] > 0) & (df["PCR"] < 20)
]



df["rank_PSR"] = df["PSR"].rank(ascending=True)
df["rank_PCR"] = df["PCR"].rank(ascending=True)

df["score"] = df["rank_PSR"] + df["rank_PCR"]



top50 = df.nsmallest(50, "score")



print("슈퍼 가치 전략 50위")
print(top50)

top50.to_excel("super_value_top50_final.xlsx", index=False)

슈퍼 가치 전략 50위
      종목코드          시가총액           매출액        영업현금흐름       PSR        PCR  \
14  267290  130767478784  1.791908e+12  7.486028e+10  0.072977   1.746821   
47  005740  101682864128  1.046896e+12  1.111586e+11  0.097128   0.914755   
38  058850   98640691200  1.042717e+12  5.344987e+10  0.094600   1.845481   
27  012800  100363780096  1.336871e+12  3.653457e+10  0.075074   2.747091   
5   010100  136570134528  1.567411e+12  5.104059e+10  0.087131   2.675716   
3   016710  139560566784  1.212355e+12  5.583065e+10  0.115115   2.499712   
22  007210  104630140928  6.430934e+11  5.455545e+10  0.162698   1.917868   
9   004140  134064439296  8.716497e+11  5.687764e+10  0.153805   2.357068   
11  002460  116321517568  6.127800e+11  1.226603e+11  0.189826   0.948323   
13  016740  129042432000  7.732586e+11  5.393377e+10  0.166881   2.392609   
29  034590  112118374400  9.629944e+11  3.909077e+10  0.116427   2.868155   
10  013570  121002385408  1.168278e+12  3.425135e+10  0.103573 

# **PER,PBR,PCR,PSR 모두 통합 후 다시 순위 측정**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd

df = pd.read_excel("/content/drive/MyDrive/DART/super_value_data.xlsx")


cols = ["PER","PBR","PSR","PCR"]
for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

#PER,PBR

df = df.dropna(subset=["PER","PBR"])

df = df[
    (df["PER"] > 0) & (df["PER"] < 20) &
    (df["PBR"] > 0) & (df["PBR"] < 3)
]

 # PSR,PCR



df["PSR"] = df["PSR"].fillna(df["PSR"].mean())
df["PCR"] = df["PCR"].fillna(df["PCR"].mean())

# 순위

df["rank_PER"] = df["PER"].rank(ascending=True)
df["rank_PBR"] = df["PBR"].rank(ascending=True)
df["rank_PSR"] = df["PSR"].rank(ascending=True)
df["rank_PCR"] = df["PCR"].rank(ascending=True)

# score 계산

df["score"] = (
    df["rank_PER"] +
    df["rank_PBR"] +
    df["rank_PSR"] +
    df["rank_PCR"]
)

# result

df = df.sort_values("score")

print(" 최종 결과")
print(df[[
    "종목코드",
    "PER","PBR","PSR","PCR",
    "rank_PER","rank_PBR","rank_PSR","rank_PCR",
    "score"
]])

# 저장

#df.to_excel("super_value_FINAL_FIXED.xlsx", index=False)

Mounted at /content/drive
🔥 최종 결과
      종목코드    PER   PBR       PSR        PCR  rank_PER  rank_PBR  rank_PSR  \
13  267290   3.98  0.27  0.072484   1.735018       3.0       2.0       1.0   
37    5740   5.51  0.30  0.097842   0.921481      13.0       5.0       5.0   
31   58850   4.72  0.46  0.092780   1.809991       7.5      17.5       4.0   
5    10100   4.72  0.35  0.086742   2.663785       7.5      11.0       3.0   
10    2460   4.79  0.30  0.198630   0.992305       9.5       5.0      17.0   
3    16710   5.59  0.31  0.114718   2.491092      14.0       7.0       7.0   
12   16740   4.43  0.51  0.166881   2.392609       5.0      21.0      13.0   
17    7210   7.04  0.35  0.161131   1.899397      20.0      11.0      12.0   
32    4840   9.22  0.26  0.138778   3.710514      25.0       1.0       9.0   
0    13520   3.99  0.52  0.077329   7.610762       4.0      22.0       2.0   
30    2450   5.36  0.34  0.405269   2.656067      11.0       9.0      24.0   
23   34590   6.13  0.46  0.114